# No-vig fair odds and expected value (EV)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JacobiusMakes/parlayapi-notebooks/blob/main/02-no-vig-and-ev.ipynb)

A sportsbook's prices always sum to more than 100% implied probability. The extra
slice is the vig (also called the margin or juice). This notebook:

1. strips the vig **two ways** (proportional and additive) to recover fair odds,
2. verifies the math against the canonical cases used by ParlayAPI's own
   [no-vig calculator](https://parlay-api.com/tools/no-vig-calculator)
   (for example -110 / -110 devigs to +100 / +100 with 4.76% vig),
3. computes the **EV of any price against the no-vig fair line**, the same
   convention as the [EV calculator](https://parlay-api.com/tools/ev-calculator),
4. runs it on live odds.

Run All is offline by default. Choose demo for a limited no-key sample, or account for your own private research.

**Default run:** offline, with no API calls or key prompts. Existing mathematical examples
are illustrative calculations, not current quotes. Select a live mode explicitly in the next cell.


In [ ]:
# Run All is offline by default. Choose demo or account explicitly for API calls.
MODE = "offline"  # "offline", "demo", or "account"
SPORT = "baseball_mlb"
RUN_EXTRA_API_CHECKS = False
RUN_POLLING = False
SAVE_PRIVATE_CSV = False

import getpass
import json
import requests

SPORTS = {"baseball_mlb", "basketball_nba", "americanfootball_nfl",
          "icehockey_nhl", "soccer_epl", "mma_mixed_martial_arts"}
BASE_URL = "https://parlay-api.com"
_runtime_key = None

def get_runtime_key():
    global _runtime_key
    if MODE != "account":
        raise RuntimeError("Choose account mode before entering a key.")
    if _runtime_key is None:
        value = getpass.getpass("Your own ParlayAPI key (hidden; account credits apply): ")
        if not value or any(ord(c) < 33 or ord(c) > 126 for c in value):
            raise RuntimeError("Enter a valid key without whitespace or control characters.")
        _runtime_key = value
    return _runtime_key

def request_json(path, *, params=None, method="GET", body=None, account=False):
    if MODE not in {"offline", "demo", "account"} or SPORT not in SPORTS:
        raise RuntimeError("Choose a listed mode and supported sport.")
    if MODE == "offline":
        raise RuntimeError("Offline mode makes no API requests.")
    if not path.startswith("/") or path.startswith("//") or ".." in path or "\\" in path:
        raise RuntimeError("Use a fixed API path.")
    headers = {"Accept": "application/json"}
    if account:
        headers["X-API-Key"] = get_runtime_key()
    try:
        with requests.request(method, "https://parlay-api.com" + path,
                              params=params, json=body, headers=headers,
                              timeout=30, allow_redirects=False, stream=True) as response:
            if response.status_code != 200:
                raise RuntimeError(f"API returned HTTP {response.status_code}. No automatic retry.")
            chunks = []
            size = 0
            for chunk in response.iter_content(65536):
                size += len(chunk)
                if size > 10_000_000:
                    raise RuntimeError("Response exceeds the size limit.")
                chunks.append(chunk)
            return json.loads(b"".join(chunks))
    except (requests.RequestException, ValueError):
        raise RuntimeError("Request or JSON response failed. No automatic retry.") from None

if MODE not in {"offline", "demo", "account"} or SPORT not in SPORTS:
    raise RuntimeError("Choose a listed mode and supported sport.")
print("Mode:", MODE)
print("Offline runs the math without network. Demo is a limited anonymous sample.")
print("Account mode prompts at runtime and uses your own allowance. Keep that copy private.")


In [ ]:
# American <-> decimal conversions, matching the conventions used by
# https://parlay-api.com/tools/no-vig-calculator and /tools/parlay-calculator.

def american_to_decimal(a):
    """+150 -> 2.5, -110 -> 1.9091. Valid American odds are >= +100 or <= -100."""
    a = float(a)
    if abs(a) < 100:
        raise ValueError(f"{a} is not a valid American price (must be >= +100 or <= -100)")
    if a > 0:
        return 1 + a / 100
    return 1 + 100 / (-a)

def decimal_to_american(d):
    """2.5 -> +150, 1.9091 -> -110 (rounded to the nearest integer)."""
    d = float(d)
    if d <= 1:
        raise ValueError(f"decimal odds must be > 1, got {d}")
    if d >= 2:
        return round((d - 1) * 100)
    return round(-100 / (d - 1))

def implied_prob(decimal_odds):
    """Implied win probability of decimal odds (includes the vig)."""
    return 1.0 / float(decimal_odds)

def fmt_american(a):
    return ("+" if a > 0 else "") + str(int(a))

## Way 1: proportional (multiplicative) devig

Convert every price to an implied probability, sum them to get the **overround**
(a number just above 1), then divide each probability by the overround. This is the
transparent, assumption-free default, and it is the method ParlayAPI's calculator
and EV tooling use.

The **vig** reported here is `overround - 1`. At -110 / -110 each side implies
52.38%, the overround is 1.0476, and the vig is 4.76%. (Some books instead quote
`(overround - 1) / overround`, which gives 4.55% for the same market. Same market,
different convention; know which one you are reading.)

In [ ]:
def devig_proportional(american_prices):
    """Proportional (multiplicative) devig of one market.

    Returns dict with raw implied probs, overround, vig, fair probs,
    fair decimal odds, and fair American odds.
    """
    decs = [american_to_decimal(a) for a in american_prices]
    raw = [1.0 / d for d in decs]
    overround = sum(raw)
    fair = [r / overround for r in raw]
    fair_dec = [1.0 / p for p in fair]
    return {
        "raw_implied": raw,
        "overround": overround,
        "vig": overround - 1,
        "fair_probs": fair,
        "fair_decimal": fair_dec,
        "fair_american": [decimal_to_american(d) for d in fair_dec],
    }

out = devig_proportional([-110, -110])
print(f"-110 / -110  ->  fair probs {out['fair_probs']}, "
      f"fair American {out['fair_american']}, vig {out['vig']*100:.2f}%")

## Way 2: additive devig

Subtract an equal share of the margin from every outcome's implied probability:
`fair_i = raw_i - (overround - 1) / n`.

Compared with proportional devig, the additive method takes the same absolute
margin off each outcome, which shifts relatively more of the vig onto longshots'
prices. On heavy longshots it can even push a probability to zero or below, so it
needs a guard. It is worth knowing because the two methods bracket what most
books actually do; power and Shin devigs (not implemented here) sit in the same
family but need extra assumptions.

In [ ]:
def devig_additive(american_prices):
    """Additive devig: remove an equal share of the margin from each outcome."""
    decs = [american_to_decimal(a) for a in american_prices]
    raw = [1.0 / d for d in decs]
    overround = sum(raw)
    share = (overround - 1) / len(raw)
    fair = [r - share for r in raw]
    if min(fair) <= 0:
        raise ValueError("additive devig produced a probability <= 0; "
                         "use proportional for markets with heavy longshots")
    fair_dec = [1.0 / p for p in fair]
    return {
        "raw_implied": raw,
        "overround": overround,
        "vig": overround - 1,
        "fair_probs": fair,
        "fair_decimal": fair_dec,
        "fair_american": [decimal_to_american(d) for d in fair_dec],
    }

# Side by side on a lopsided two-way market:
prices = [-200, +170]
p = devig_proportional(prices)
a = devig_additive(prices)
print(f"market {prices}, vig {p['vig']*100:.2f}%")
print(f"  proportional fair: {[round(x, 4) for x in p['fair_probs']]} "
      f"-> {[fmt_american(x) for x in p['fair_american']]}")
print(f"  additive fair:     {[round(x, 4) for x in a['fair_probs']]} "
      f"-> {[fmt_american(x) for x in a['fair_american']]}")

## Verify against the site calculator's canonical cases

These are the exact test vectors the [no-vig calculator](https://parlay-api.com/tools/no-vig-calculator)
self-tests on load. If any assert fires, the math above has drifted.

In [ ]:
CANONICAL = [
    # (american prices, vig, fair probs, fair american)
    ([-110, -110],      0.0476, [0.5, 0.5],               [100, 100]),
    ([-120, +100],      0.0455, [0.5217, 0.4783],         [-109, 109]),
    ([+150, -170],      0.0296, [0.3885, 0.6115],         [157, -157]),
    ([+150, +220, +240],0.0066, [0.3974, 0.3104, 0.2922], [152, 222, 242]),
    ([-110, +100],      0.0238, [0.5116, 0.4884],         [-105, 105]),
    ([-200, +170],      0.0370, [0.6429, 0.3571],         [-180, 180]),
]

for prices, vig, fair, fair_am in CANONICAL:
    got = devig_proportional(prices)
    assert abs(got["vig"] - vig) < 5e-5, (prices, got["vig"])
    for g, want in zip(got["fair_probs"], fair):
        assert abs(g - want) < 5e-5, (prices, got["fair_probs"])
    assert got["fair_american"] == fair_am, (prices, got["fair_american"])

print(f"all {len(CANONICAL)} canonical devig cases pass")
assert devig_proportional([-110, -110])["fair_american"] == [100, 100]
print("-110/-110 -> +100/+100 with vig "
      f"{devig_proportional([-110, -110])['vig']*100:.2f}% (the textbook case)")

## EV of a price against the no-vig fair line

Once you have a fair win probability `p`, the expected value of a price per $1
staked is:

```
EV per $1 = p * (dec - 1) - (1 - p)
```

where `dec` is the decimal odds of the price you can actually bet. Multiply by 100
for EV percent, or by your stake for dollars. Positive means the bet wins more, on
average, than it risks. This is the exact formula behind the
[EV calculator](https://parlay-api.com/tools/ev-calculator).

In [ ]:
def ev_per_dollar(fair_prob, american_price):
    dec = american_to_decimal(american_price)
    return fair_prob * (dec - 1) - (1 - fair_prob)

# Example: the market says -110/-110 (fair 50%), but one book hangs +105 on a side.
fair = devig_proportional([-110, -110])["fair_probs"][0]
ev = ev_per_dollar(fair, +105)
print(f"fair p = {fair:.4f}, price +105  ->  EV {ev*100:+.2f}% of stake")
assert abs(ev - 0.025) < 1e-9  # 0.5 * 1.05 - 0.5

## Run it on live odds

For each event we devig every book's moneyline separately, average the fair
probabilities across books into a simple consensus, then score every available
price against that consensus. Rows at the top are the closest to +EV right now.

Honest caveats: without a key the demo feed carries only a few books, so the
consensus is thin; a real workflow devigs a sharp book (or many books) and knows
that a fair price is the market's opinion, not the truth.

In [ ]:
def fetch_odds(sport=None, markets="h2h,spreads,totals", odds_format="american"):
    """A chosen demo or account request. Offline returns no live observations."""
    sport = sport or SPORT
    if sport not in SPORTS or odds_format != "american":
        raise RuntimeError("Choose a supported sport and American odds.")
    requested = markets.split(",")
    if not requested or any(m not in {"h2h", "spreads", "totals"} for m in requested):
        raise RuntimeError("Choose h2h, spreads, or totals.")
    if MODE == "offline":
        return []
    if MODE == "account":
        events = request_json(f"/v1/sports/{sport}/odds", account=True,
                              params={"markets": markets,
                                      "oddsFormat": "american"})
    else:
        payload = request_json(f"/v1/try/{sport}/odds")
        if (not isinstance(payload, dict) or payload.get("demo") is not True
                or not isinstance(payload.get("events"), list) or len(payload["events"]) > 5):
            raise RuntimeError("Unexpected demo response. No observations used.")
        events = payload["events"]
    if not isinstance(events, list) or any(not isinstance(e, dict) or e.get("sport_key") != sport for e in events):
        raise RuntimeError("Response does not match the chosen sport.")
    return events


In [ ]:
import pandas as pd

try:
    events = fetch_odds()
except Exception:
    events = []
    print(f"Fetch failed (request failed). Check your connection or key and re-run this cell.")
rows = []
for ev in events:
    label = f"{ev['away_team']} at {ev['home_team']}"
    for bm in ev.get("bookmakers", []):
        for mkt in bm.get("markets", []):
            if mkt["key"] != "h2h":
                continue
            outs = mkt.get("outcomes", [])
            if len(outs) < 2:
                continue
            try:
                devig = devig_proportional([o["price"] for o in outs])
            except ValueError:
                continue
            for o, fair_p in zip(outs, devig["fair_probs"]):
                rows.append({"event": label, "outcome": o["name"], "bookmaker": bm["key"],
                             "price": o["price"], "fair_prob_this_book": fair_p})

odds = pd.DataFrame(rows)
if odds.empty:
    print("No h2h observations loaded. Offline does not fetch data; live responses can be empty.")
else:
    consensus = (odds.groupby(["event", "outcome"])["fair_prob_this_book"]
                     .mean().rename("consensus_fair_prob").reset_index())
    odds = odds.merge(consensus, on=["event", "outcome"])
    odds["ev_pct"] = [
        100 * ev_per_dollar(p, price)
        for p, price in zip(odds["consensus_fair_prob"], odds["price"])
    ]
    n_books = odds["bookmaker"].nunique()
    print(f"{len(odds)} prices from {n_books} books, scored against a {n_books}-book consensus")
    display(odds.sort_values("ev_pct", ascending=False).head(10).round(4))

## Cross-check with the hosted EV scanner

ParlayAPI also runs this scan server-side across its whole book list, anchored on a
sharp book instead of a thin consensus. The keyless demo version is
`GET /v1/try/{sport}/ev` (same demo envelope pattern: results are nested inside
demo metadata). With a key, use `GET /v1/sports/{sport}/ev`.

In [ ]:
if MODE == "offline" or not RUN_EXTRA_API_CHECKS:
    print("Optional API cross-check is off. The local math above remains runnable.")
else:
    try:
        path = f"/v1/sports/{SPORT}/ev" if MODE == "account" else f"/v1/try/{SPORT}/ev"
        payload = request_json(path, account=MODE == "account")
        opps = payload.get("opportunities", []) if isinstance(payload, dict) else payload
        print("API cross-check returned", len(opps), "rows. Inspect only in your private runtime.")
    except RuntimeError:
        print("API cross-check failed. No automatic retry; see https://parlay-api.com/docs.")


---

**More ParlayAPI resources**

- Docs: [parlay-api.com/docs](https://parlay-api.com/docs)
- Free API key (no card): [parlay-api.com/signup](https://parlay-api.com/signup)
- Browser calculators the math here matches: [no-vig](https://parlay-api.com/tools/no-vig-calculator), [parlay](https://parlay-api.com/tools/parlay-calculator), [EV](https://parlay-api.com/tools/ev-calculator)
- The rest of this series: [github.com/JacobiusMakes/parlayapi-notebooks](https://github.com/JacobiusMakes/parlayapi-notebooks)

These notebooks are for personal and internal research and education. Nothing here is betting advice.

## Private runtime data

Keep this notebook's code shareable and your account work private. Do not paste keys into
code cells, save them in notebook text, or commit downloaded observations. The hidden prompt
keeps the key in this runtime only. Clear all outputs before sharing or saving to GitHub;
Colab's output-omission setting is an additional safeguard, not a guarantee on other hosts.
The original published notebook contains no saved API results. Do not share an executed
account notebook or its exports. Each person uses their own account and key.

The MIT license covers code. API access does not grant public redisplay or redistribution
rights. Your applicable [terms](https://parlay-api.com/terms) and written agreement govern data.
Current coverage and plans: [docs](https://parlay-api.com/docs), [pricing](https://parlay-api.com/pricing).


In [ ]:
# Drop the runtime reference when finished. Restart the runtime to release other state.
_runtime_key = None
